# Preface

2022/06/12

Compiled from labuladong


The basic storage methods for data structures are just linked and sequential; the basic operations are insert, delete, search, update; and traversal methods are just iteration and recursion.

The essence of algorithms is "exhaustive search."

Difficulties of exhaustive search:
- No omissions,
- No redundancy: search smartly, using as few resources as possible to find the answer.

# Array / Linked List

- Traversal

- Two pointers
    - Sliding window
- Prefix sum
- Difference array

## Traversal

For any recursive traversal, there's always a preorder position and a postorder position.

In [ ]:
# Iterative array traversal

def traverse(nums):
    for num in nums:
        ...
    return ...

# Recursive array traversal

def traverse(nums, i):
    if i == len(nums):
        ...
        return
    
    # preorder position operations
    ...
    traverse(nums, i + 1)
    
    # postorder position operations
    ...
    return ...

# Iterative linked-list traversal
def traverse(head):
    p = head
    while p:
        # ...
        p = p.next
        
    return ...

# Recursive linked-list traversal
def traverse(head):
    if head is None:
        return ...
    
    # preorder position operations
    ...
    traverse(head.next)
    
    # postorder position operations
    ...
    return ...
    


## Two Pointers

### Sliding Window

Key points:
- How should the window be updated when a new item is added?
- When should the window be shrunk?
- What values should be updated when shrinking the window?

Uses a half-open interval [left, right) — the benefit is that when l=r=0, the window contains no values, corresponding to the initial state.

In [ ]:
# Sliding window template

def slidingWindow(s, t):
    l = 0
    r = 0
    res = ...
    while r < len(s):
        item = s[r]
        r += 1
        # update the window
        # ....
        
        print("Debug: window[{}: {}]".format(l, r))
        
        while needsShrink(window):
            removed = s[l]
            l += 1
            
            # update the window
            # ...
    return res...


## Prefix Sum Technique

Applicable when: the original array won't be modified, and you need to frequently compute the sum over some range

Example problem: 304

Template: two operations — initialization and query.

There are two ways to implement it:
- One with padding: no need to worry about the starting position, at the cost of a bit more space. When handling indices, pay extra attention — there's one extra slot, and presum[i] represents the sum of the first i elements
- One without padding: saves space, but requires special handling at the edges

In [ ]:
# Prefix sum template

# 1D version with padding
class PreSum(object):
    def __init__(self, nums):
        self.prefix = [0 for _ in range(len(nums)+1)] # one extra slot here — padding space that reduces the need for special-case handling

        for i in range(len(nums)):
            self.prefix[i+1] = self.prefix[i]+nums[i]
            
    def query(self, i, j):
        # by default, i <= j and it is the closed sublist
        return self.prefix[j+1] - self.prefix[i] # compute using j+1


# 1D version without padding
class PreSum(object):
    def __init__(self, nums):
        self.prefix = [0 for _ in range(len(nums))]
        # handle the first element separately
        self.prefix[0] = nums[0]
        
        # repeat for the others
        for i in range(1, len(nums)):
            self.prefix[i] = self.prefix[i-1]+nums[i]
            
    def query(self, i, j):
        # by default, i <= j and it is the closed sublist
        
        if i == 0: # need to handle the case where the query starts at 0 separately
            return self.prefix[j]
        return self.prefix[j] - self.prefix[i-1]

# 2D version with padding
class NumMatrix:

    def __init__(self, matrix: List[List[int]]):
        self.m = len(matrix)
        self.n = len(matrix[0])
        self.presum = [[0 for _ in range(self.n+1)] for _ in range(self.m+1)] # one extra row/column of padding space
        
        for l in range(self.m):
            for c in range(self.n):
                # note: indices into presum all need +1
                self.presum[l+1][c+1] = self.presum[l][c+1] + self.presum[l+1][c] - self.presum[l][c] + matrix[l][c]

    def sumRegion(self, row1: int, col1: int, row2: int, col2: int) -> int:
        # print(row1, col1, row2, col2)
        return self.presum[row2+1][col2+1] - self.presum[row1][col2+1] - self.presum[row2+1][col1] + self.presum[row1][col1]



# 2D version without padding
class NumMatrix:
    def __init__(self, matrix: List[List[int]]):
        self.m = len(matrix)
        self.n = len(matrix[0])
        self.presum = [[0 for _ in range(self.n)] for _ in range(self.m)]
        # initialization
        # need to initialize the top-left element separately
        self.presum[0][0] = matrix[0][0]
        # need to initialize the first row's elements separately
        for c in range(1, self.n):
            self.presum[0][c] = self.presum[0][c-1] + matrix[0][c]
        # need to initialize the first column's elements separately
        for l in range(1, self.m):
            self.presum[l][0] = self.presum[l-1][0] + matrix[l][0]
        # calculate for others
        for l in range(1, self.m):
            for c in range(1, self.n):
                self.presum[l][c] = self.presum[l-1][c] + self.presum[l][c-1] - self.presum[l-1][c-1] + matrix[l][c]

    def sumRegion(self, row1: int, col1: int, row2: int, col2: int) -> int:
        # need to handle the top-left corner, first row, and first column separately
        if row1 == 0 and col1 == 0:
            return self.presum[row2][col2]
        if row1 == 0:
            return self.presum[row2][col2] - self.presum[row2][col1-1]
        if col1 == 0:
            return self.presum[row2][col2] - self.presum[row1-1][col2]

        return self.presum[row2][col2] - self.presum[row1-1][col2] - self.presum[row2][col1-1] + self.presum[row1-1][col1 - 1]



## Difference Array Technique

Applicable when: frequently incrementing/decrementing values over sub-ranges

Template: only need to modify the value at the start of the range and the position right after the end of the range

Three operations: initialization, modification, and query

Example problem: 1109. An interesting variant occurred to me: suppose each flight has a maximum number of 💺 seats, and there's a stream of bookings — for each booking, return whether it succeeded. A booking succeeds if, for every flight, the total number of booked seats never exceeds its capacity. If a booking fails, stop processing the remaining bookings.

How would you solve this variant? Better not use a difference array here — instead, track the seat count for each flight directly. Why? Because each booking requires checking the state of every flight within the booking's range, and that step's complexity can't be avoided — but avoiding exactly that complexity is precisely what a difference array is good for.

In [ ]:
# Template
class DiffNums(object):
    def __init__(self, nums):
        self.len = len(nums)
        self.diff = [num for _ in nums]
        for i in range(1, self.len):
            self.diff[i] = nums[i] - nums[i-1]
            
    def incremental(self, i, j, val):
        self.diff[i] += val
        if j +1 < self.len:
            self.diff[j+1] -= val
        
    def query(self):
        res = []
        if self.len > 0:
            res = [self.diff[0]]
            for i in range(1, self.len):
                res.append(res[-1] + self.diff[i])
        return res
        

## Monotonic Stack

The key is figuring out exactly when to push and when to pop.

In [ ]:
# Monotonic stack template
def nextGreaterElement(nums):
    n = len(nums)
    res = [0 for num in nums]
    s = []
    for i in range(n-1, -1, -1):
        
        while s and s[-1] <= nums[i]:
            s.pop()
            
        res[i] = -1 if not s else s[-1]
        s.append(nums[i])
    return res

# Binary Tree

Recursive solutions for binary trees fall into two categories of approach:
- Traverse once and derive the answer: backtracking. Implemented with a `traverse` function plus external variables. The **"traversal"** mindset.
- Decompose the problem and compute the answer: dynamic programming / recursion. Derive the answer to the original problem from the answers to sub-problems (subtrees). Make full use of the function's return value — the **"decompose the problem"** mindset.

Regardless of which mode, you need to think about:

If you isolate a single binary tree node,
- What does it need to do?
- When does it need to do it (pre/in/postorder position)?
    - Code at the preorder position: can only access data passed down from the parent node via function arguments
    - Code at the postorder position: can access not just the argument data, but also data passed back up from subtrees via function return values
        - Once you notice a problem involves subtrees, it's very likely you'll need to give the function a sensible definition and return value, and write the logic at the postorder position.


Every binary tree problem is really about injecting clever logic at the pre/in/postorder positions to achieve your goal. Just think carefully about what each individual node should do!

In [ ]:
def traverse(node):
    # base case 
    # ...
    
    # preorder operations
    # ...
    
    traverse(node.left)
    
    # inorder operations
    # ...
    
    traverse(node.right)
    
    # postorder operations
    # ...
    return ...

# Binary Search

A useful technique when analyzing binary search: avoid `else` — instead spell out every case with `else if`, so all the details are clearly laid out

Applicable scenario: abstract the problem into an independent variable `x`, a function `f(x)` of `x`, and a target value `target`, where the three satisfy:

1. `f(x)` is a monotonic function of `x`
2. The problem asks you to compute the value of `x` that satisfies the constraint `f(x) == target`.

Common problem type: minimizing the maximum, etc. — convert into a decision problem and binary-search the answer.

Solution steps:
1. Determine what `x`, `f(x)`, and `target` are, and write the code for `f(x)`
2. Find the range of `x`, use it as the search interval for binary search, and initialize the `left` and `right` variables
3. Based on the problem's requirements, determine whether to use the left-boundary or right-boundary binary search variant, and write the code.

In [ ]:
# Binary search template

def binarySearch(nums, target):
    l = 0
    r = ...
    while ...:
        m = l + (r-l)//2 # avoid m = (l + r) // 2, to prevent overflow
        if nums[m] == target:
            ...
        elif nums[m] < target:
            ...
        elif nums[m] > target:
            ...
    return ...

# Binary search for the left boundary
def left_bound(nums, target):
    left = 0
    right = len(nums)
    
    while left < right:
        mid = left + (right - left) // 2
        if nums[mid] == target:
            right = mid
        elif nums[mid] < target:
            left = mid + 1
        else:
            # nums[mid] > target:
            right = mid
    return left
        
# Binary search for the left boundary, abstracted version
def left_bound(nums, target):
    left = 0
    right = len(nums)
    
    while left < right:
        mid = left + (right - left) // 2
        mid_val = f(mid, nums)
        if mid_val == target:
            right = mid
        elif mid_val < target:
            left = mid + 1
        else:
            # mid_val > target:
            right = mid
    return left

# Binary search for the right boundary
def right_bound(nums, target):
    left = 0
    right = len(nums)
    
    while left < right:
        mid = left + (right - left) // 2
        if nums[mid] == target:
            left = mid + 1
        elif nums[mid] < target:
            left = mid + 1
        else:
            # nums[mid] > target:
            right = mid
    return left - 1

In [ ]:
# Binary search problem-solving template

def f(x):
    ...
    
def solution(nums, target):
    left = 0
    right = len(nums)
    while left < right:
        if f(mid) == target:
            # ask yourself: are we looking for the left boundary or the right boundary?
            ...
        elif f(mid) > target:
            # ask yourself: how do I make f(x) smaller?
        else:
            #  if f(mid) < target:
            # ask yourself: how do I make f(x) larger?
    return left

# Graph Theory

## Minimum Spanning Tree

- Kruskal's algorithm: first sort all the edges, then, starting from the smallest-weight edge, pick edges that belong to the minimum spanning tree (guaranteed via Union-Find), building up the MST
    - Union-Find: with a small tweak to `findRoot`, complexity can be optimized to O(1)
- Prim's algorithm: (a dynamic version of Kruskal's, using a priority queue and the cut property) starting from the cut at one vertex, run BFS-like logic, each time adding to the MST an edge whose (outward) endpoint is not yet in the MST, growing the tree

## Shortest Path Weight

Dijkstra's algorithm: given a graph and a start vertex, returns an array recording the shortest path weight to each vertex
- Dijkstra uses a priority queue mainly as an efficiency optimization — the idea is similar to a greedy algorithm.
- If the destination is known in advance, an optimization is: stop traversal as soon as the destination is reached for the first time.

In [ ]:
# Dijkstra
import heapq
def Dijkstra(start, end, graph):
    pq = []
    distTo = {start: 0} # don't forget to record the weight for start!
    
    for end_node, edge_weight in graph[start]:
        distTo[end_node] = edge_weight
        heapq.heappush(pq, (edge_weight, end_node))
    while pq:
        edge_weight, end_node = heapq.heappop(pq)
        if end_node == end:
            return distTo[end_node]
        for neigh, weight in graph[end_node]:
            if neigh not in distTo or distTo[neigh] > distTo[end_node] + weight:
                distTo[neigh] = distTo[end_node] + weight
                heapq.heappush(pq, (weight, neigh))
    
    

# Backtracking

Backtracking is essentially the pre/postorder traversal problem for an N-ary tree.

Unlike dynamic programming, which optimizes by exploiting overlapping subproblems, backtracking is pure brute-force exhaustive search, and its complexity is generally very high.

- When writing a backtracking algorithm, you need to maintain the **path** taken so far and the current **list of choices** available; when the **termination condition** is triggered, record the **path** into the result set.
- Sometimes you don't need every valid answer — you just want one answer. What then? Return early at the right point to cut the search short.

In [ ]:
# backtrack
result = []
def backtrack(path, options):
    if satisfyEnd(path):
        result.add(path)
        ...
        
    for op in options:
        makeOption(op)
        # update path and options
        backtrack(path, options)
        # update path and options
        revertOption(op)
        

# Dynamic Programming

1. First write the brute-force exhaustive solution (the state transition equation)
2. Add a memo and it becomes top-down recursion
3. Tweak it and it becomes bottom-up iteration
4. Optimize the space complexity
